# Stichproben und Konfidenzintervalle

**Notebook 2 von 5.** `01_distributions.ipynb` hat **einzelne Pinguine** beschrieben: wie das
Körpergewicht von 68 Zügelpinguinen (Chinstrap) um ihren Mittelwert von 3.733 g streut. Dieses
Notebook fragt nach diesem Mittelwert selbst.

Diese 68 Pinguine sind nicht alle Zügelpinguine, die es gibt. Sie sind eine **Stichprobe**, und
3.733 g ist nur der Durchschnitt dieser einen Stichprobe. Wiegst du 68 andere, bekommst du einen
anderen Durchschnitt. Wie weit kann 3.733 g also vom wahren Durchschnitt aller Zügelpinguine
entfernt sein?

Die Antwort ist die zweite Idee, auf der jeder Hypothesentest aufbaut, der **Standardfehler**,
und er liefert dir ein **Konfidenzintervall**: einen Bereich plausibler Werte für den wahren
Durchschnitt. Am Ende des Notebooks beantwortet dieser Bereich bereits eine Ja-Nein-Frage, und
genau das macht ein Test in `03_how_a_test_works.ipynb`.

Am Ende solltest du Folgendes können:

- einen Parameter der Grundgesamtheit von einer Stichprobenstatistik unterscheiden.
- per Simulation zeigen, wie stark ein Stichprobendurchschnitt schwankt und warum eine größere
  Stichprobe weniger schwankt.
- den Standardfehler eines Mittelwerts aus einer einzigen Stichprobe berechnen.
- ein 95-%-Konfidenzintervall für einen Mittelwert bilden und sagen, worauf sich die 95 %
  beziehen.
- mit einem Konfidenzintervall beurteilen, ob ein behaupteter Wert plausibel ist.

Zuerst die Bibliotheken, dann dieselben Zügelpinguine wie in Notebook 01.

In [ ]:
# Datenanalyse
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt
import seaborn as sns

# Statistik
from scipy import stats

sns.set_style("whitegrid")

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")

# die 68 Zügelpinguine und ihr Körpergewicht
chinstrap = df_penguins[df_penguins["species"] == "Chinstrap"]
mass = chinstrap["body_mass_g"]

## 1. Grundgesamtheit und Stichprobe

Zwei Begriffe tragen dieses ganze Notebook.

- Die **Grundgesamtheit** (Population) sind alle Individuen, über die du eine Aussage treffen
  willst: *alle* Zügelpinguine. Du kannst sie fast nie vollständig messen.
- Eine **Stichprobe** ist der Teil, den du tatsächlich gemessen hast: unsere 68.

Eine Zahl, die die Grundgesamtheit beschreibt, ist ein **Parameter**. Sie ist fest, aber
unbekannt. Dieselbe Zahl, berechnet auf der Stichprobe, ist eine **Statistik**. Sie ist bekannt,
würde aber bei einer anderen Stichprobe anders ausfallen. In der Statistik bekommen beide
unterschiedliche Buchstaben, damit man sie nie verwechselt:

| | Parameter der Grundgesamtheit | Stichprobenstatistik |
| :--- | :---: | :---: |
| Mittelwert | $\mu$ | $\bar{x}$ |
| Standardabweichung | $\sigma$ | $s$ |
| Anzahl der Individuen | | $n$ |

Unsere Stichprobenstatistiken sind $\bar{x} = 3.733$ g, $s = 384$ g und $n = 68$. Die Frage
dieses Notebooks lautet, wie weit $\bar{x}$ von $\mu$ entfernt sein kann.

## 2. Wie stark wackelt ein Durchschnitt?

Wir können nicht für weitere 68 Pinguine zurück in die Antarktis, aber ein Computer kann das
Nächstbeste tun. **Tun wir so, als kennten wir die Grundgesamtheit**: Angenommen, alle
Zügelpinguine haben wirklich ein mittleres Körpergewicht von 3.750 g und eine Standardabweichung
von 400 g. Dann ziehen wir daraus immer wieder Stichproben von 68 und schauen, wie stark ihre
Durchschnitte schwanken.

> **Hinweis.** Die 3.750 g und 400 g sind für die Simulation erfunden, runde Zahlen nahe an
> unserer Stichprobe. Entscheidend ist, dass *wir* sie gewählt haben. So wissen wir genau, was
> die richtige Antwort ist, und können sehen, wie nah jede Stichprobe ihr kommt.

In [ ]:
# ein Zufallszahlengenerator mit festem Seed, damit die Ergebnisse bei jedem Lauf gleich sind
rng = np.random.default_rng(42)

# die angenommene Grundgesamtheit
mu = 3750
sigma = 400

# eine Stichprobe von 68 Pinguinen und ihr Durchschnitt
one_sample = rng.normal(loc=mu, scale=sigma, size=68)
print(f"eine Stichprobe von 68: Durchschnitt {one_sample.mean():.0f} g")

# 1.000 Stichproben von 68 auf einmal: eine Tabelle mit 1.000 Zeilen, eine Stichprobe pro Zeile
samples = rng.normal(loc=mu, scale=sigma, size=(1000, 68))
sample_means = samples.mean(axis=1)  # der Durchschnitt jeder Zeile
print(f"1.000 Stichproben von 68: Durchschnitte von {sample_means.min():.0f} g "
      f"bis {sample_means.max():.0f} g")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(sample_means, bins=30, ax=ax)
ax.axvline(mu, color="crimson", linestyle="--", linewidth=2, label=f"wahrer Mittelwert, {mu} g")
ax.set_title("Die Durchschnitte von 1.000 Stichproben mit je 68 Pinguinen")
ax.set_xlabel("durchschnittliches Körpergewicht der Stichprobe (g)")
ax.legend()
plt.show()

Dieses Histogramm zeigt keine Pinguine. Jeder Balken zählt **Stichprobendurchschnitte**, und
die Form, die sie bilden, hat einen Namen: die **Stichprobenverteilung des Mittelwerts**. Drei
Dinge daran sind wichtig.

1. **Sie ist um den wahren Mittelwert zentriert.** Manche Stichproben fallen schwer aus, manche
   leicht, aber im Mittel liegen sie richtig.
2. **Sie ist viel schmaler als die Pinguine selbst.** Einzelne Zügelpinguine streuen über weit
   mehr als tausend Gramm, aber Durchschnitte von 68 landen meist innerhalb von etwa hundert
   Gramm um 3.750. Beim Mitteln heben sich die schweren und die leichten auf.
3. **Sie ist glockenförmig.** Das ist wieder die Normalverteilungskurve aus Notebook 01, also
   gelten alle ihre Werkzeuge: Flächen, `ppf` und 1,96.

Die Streuung dieser Verteilung, also die Standardabweichung der Stichprobendurchschnitte, ist
der **Standardfehler**. Er misst, wie stark ein Durchschnitt von einer Stichprobe zur nächsten
wackelt.

In [ ]:
print(f"Standardfehler, gemessen an den 1.000 Durchschnitten: {sample_means.std():.1f} g")

### Größere Stichproben wackeln weniger

Der Standardfehler hängt von der Stichprobengröße ab. Wiederhole die Simulation mit Stichproben
von 10, 68 und 300 Pinguinen:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

for ax, size in zip(axes, [10, 68, 300]):
    # 1.000 Stichproben dieser Größe und der Durchschnitt jeder einzelnen
    means = rng.normal(loc=mu, scale=sigma, size=(1000, size)).mean(axis=1)

    sns.histplot(means, bins=30, ax=ax)
    ax.axvline(mu, color="crimson", linestyle="--", linewidth=2)
    ax.set_title(f"n = {size}: Standardfehler {means.std():.0f} g")
    ax.set_xlabel("Stichprobendurchschnitt (g)")

plt.tight_layout()
plt.show()

Je größer die Stichprobe, desto enger liegen die Durchschnitte. Der Zusammenhang ist exakt, und
er liefert die Formel, die du ab jetzt verwendest:

$$
\text{Standardfehler} = \frac{\sigma}{\sqrt{n}}
$$

Für $n = 68$ sind das $400 / \sqrt{68} \approx 48{,}5$ g, nahe an den 50 g, die die 1.000
Stichproben gemessen haben. Sie stimmen nicht exakt überein, weil 1.000 Stichproben selbst nur
eine Stichprobe aller möglichen Stichproben sind. Die Wurzel ist der Grund, warum Präzision
teuer ist: **Die vierfache Datenmenge halbiert den Standardfehler nur.**

### Selbst wenn die Grundgesamtheit nicht glockenförmig ist

In Abschnitt 4 von Notebook 01 war das Körpergewicht aller Pinguine zusammengenommen schief, mit
einem zweiten Buckel, und eine Normalverteilungskurve passte schlecht dazu. Nimm jetzt stattdessen
Durchschnitte davon.

In [ ]:
# jeder Pinguin mit erfasstem Körpergewicht, alle Arten zusammen
all_mass = df_penguins["body_mass_g"].dropna()

# 1.000 Stichproben von 68, gezogen aus diesen 342 Pinguinen, und der Durchschnitt jeder einzelnen
pooled_means = rng.choice(all_mass, size=(1000, 68)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(all_mass, bins=25, ax=axes[0])
axes[0].set_title("Einzelne Pinguine: schief")
axes[0].set_xlabel("Körpergewicht (g)")

sns.histplot(pooled_means, bins=30, ax=axes[1])
axes[1].set_title("Durchschnitte von 68 Pinguinen: glockenförmig")
axes[1].set_xlabel("durchschnittliches Körpergewicht der Stichprobe (g)")

plt.tight_layout()
plt.show()

Die einzelnen Werte sind schief, ihre Durchschnitte nicht. Das ist der **zentrale
Grenzwertsatz**: Bei einer ausreichend großen Stichprobe sind Durchschnitte annähernd
normalverteilt, egal welche Form die Daten haben, aus denen sie stammen. Eine gängige Faustregel
lautet: 30 oder mehr.

Deshalb funktionieren Tests über Durchschnitte so breit. Es ist auch der Grund, warum man in
Notebook 04 einem t-Test selbst bei schiefen Restaurantrechnungen noch vertrauen kann.

## 3. Der Standardfehler aus einer einzigen Stichprobe

Im echten Leben gibt es nur eine Stichprobe, und das $\sigma$ der Grundgesamtheit ist unbekannt.
Also schätzen wir es mit der Standardabweichung der Stichprobe selbst, $s$:

$$
SE = \frac{s}{\sqrt{n}}
$$

Diese eine Zeile sagt dir anhand von 68 Pinguinen, wie stark ihr Durchschnitt wackeln würde,
wenn du die ganze Expedition wiederholen könntest.

In [ ]:
n = len(mass)
mean_mass = mass.mean()
std_mass = mass.std()

# der Standardfehler des Mittelwerts, geschätzt aus dieser einen Stichprobe
se_mass = std_mass / np.sqrt(n)

print(f"Stichprobendurchschnitt : {mean_mass:.1f} g")
print(f"Standardfehler          : {se_mass:.1f} g")

Unser Durchschnitt von 3.733 g hat einen Standardfehler von etwa 47 g. Merk dir beide Zahlen:
Der nächste Abschnitt macht daraus einen Bereich, und Notebook 03 macht daraus einen Test.

## 4. Ein 95-%-Konfidenzintervall

Die Stichprobenverteilung ist eine Normalverteilungskurve, und Notebook 01 hat gezeigt, dass die
mittleren 95 Prozent jeder Normalverteilungskurve innerhalb von 1,96 Standardabweichungen um ihr
Zentrum liegen. Die Standardabweichung *dieser* Kurve ist der Standardfehler. Also ist

$$
\bar{x} \pm 1{,}96 \times SE
$$

ein Bereich, der den wahren Mittelwert bei 95 Prozent der Stichproben einfängt. Dieser Bereich
ist ein **95-%-Konfidenzintervall**.

In [ ]:
# die Normalverteilungsversion: der Stichprobendurchschnitt plus und minus 1,96 Standardfehler
z_crit = stats.norm.ppf(0.975)
print(f"normal: {mean_mass - z_crit * se_mass:.0f} g bis {mean_mass + z_crit * se_mass:.0f} g")

### Eine Korrektur: die t-Verteilung

Es gibt einen Haken. Wir kannten $\sigma$ nicht, also haben wir es mit $s$ geschätzt, und diese
Schätzung wackelt selbst auch. Die **t-Verteilung** aus Abschnitt 5 von Notebook 01 berücksichtigt
das. Wie du dort gesehen hast, sieht sie aus wie die Normalverteilungskurve mit etwas schwereren
Rändern, und wie schwer sie sind, hängt von den **Freiheitsgraden** ab, hier $n - 1 = 67$. Je
weniger Pinguine, desto schwerer die Ränder und desto breiter muss das Intervall sein, um ehrlich
zu bleiben.

In [ ]:
x = np.linspace(-4, 4, 400)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, stats.norm.pdf(x), color="black", linewidth=2, label="Normalverteilung")
ax.plot(x, stats.t.pdf(x, df=4), color="crimson", linestyle="--", label="t, 4 Freiheitsgrade")
ax.plot(x, stats.t.pdf(x, df=67), color="steelblue", linestyle=":", linewidth=2,
        label="t, 67 Freiheitsgrade")
ax.set_title("Die t-Verteilung: die Normalverteilungskurve mit schwereren Rändern")
ax.set_xlabel("Standardfehler vom Zentrum entfernt")
ax.legend()
plt.show()

print(f"Grenzwert normal   : {stats.norm.ppf(0.975):.3f}")
print(f"Grenzwert t, df=67 : {stats.t.ppf(0.975, df=n - 1):.3f}")

Mit 67 Freiheitsgraden ist die t-Verteilung fast die Normalverteilung: Der Grenzwert liegt bei
1,996 statt 1,96. Mit nur 5 Pinguinen sähe das ganz anders aus. Die t-Version ist immer dann die
richtige, wenn die Standardabweichung aus der Stichprobe stammt, also fast immer, und scipy
baut das ganze Intervall mit einem einzigen Aufruf:

In [ ]:
# 95-%-Konfidenzintervall für den Mittelwert, aus der t-Verteilung
ci_lower, ci_upper = stats.t.interval(0.95, df=n - 1, loc=mean_mass, scale=se_mass)
print(f"95-%-Konfidenzintervall für den Durchschnitt der Zügelpinguine: {ci_lower:.0f} g bis {ci_upper:.0f} g")

Das durchschnittliche Körpergewicht der Zügelpinguine liegt also plausibel irgendwo zwischen
etwa 3.640 g und 3.826 g. Genauer können es unsere 68 Pinguine nicht eingrenzen.

> **Übung:**
>
> Bilde ein 95-%-Konfidenzintervall für die durchschnittliche **Flossenlänge** der
> Zügelpinguine.

<details><summary>
Hier klicken für einen Tipp ...
</summary>

In `stats.t.interval` gehen drei Zahlen ein: die Freiheitsgrade `n - 1`, der
Stichprobendurchschnitt als `loc` und der Standardfehler `s / np.sqrt(n)` als `scale`.
</details>

In [ ]:
# dein Code: 95-%-Konfidenzintervall für die Flossenlänge der Zügelpinguine

## 5. Was die 95 % bedeuten

Es ist verlockend, das Intervall so zu lesen: *„Mit 95 Prozent Wahrscheinlichkeit liegt der wahre
Durchschnitt zwischen 3.640 g und 3.826 g.“* Das stimmt nicht ganz, und der Unterschied ist für
das Testen wichtig.

Der wahre Durchschnitt ist eine feste Zahl. Entweder liegt er in diesem Intervall oder nicht; da
ist kein Zufall mehr im Spiel. Die 95 Prozent beschreiben die **Methode**: Bildest du Intervalle
auf diese Weise für Stichprobe um Stichprobe, fangen 95 Prozent davon den wahren Mittelwert ein.
Unser Intervall ist eines davon, und wir können nicht sagen, ob es zu den glücklichen 95 oder den
unglücklichen 5 gehört.

Mit der angenommenen Grundgesamtheit aus Abschnitt 2 können wir diese Behauptung überprüfen, denn
dort wissen wir, dass der wahre Mittelwert 3.750 g beträgt.

In [ ]:
# 1.000 neue Stichproben von 68 aus der angenommenen Grundgesamtheit (mu = 3750, sigma = 400)
samples = rng.normal(loc=mu, scale=sigma, size=(1000, 68))

# ein 95-%-Intervall pro Stichprobe
means = samples.mean(axis=1)
standard_errors = samples.std(axis=1, ddof=1) / np.sqrt(68)
lowers, uppers = stats.t.interval(0.95, df=67, loc=means, scale=standard_errors)

# hat jedes Intervall den wahren Mittelwert eingefangen?
caught = (lowers <= mu) & (mu <= uppers)
print(f"{caught.sum()} von 1.000 Intervallen enthalten den wahren Mittelwert ({caught.mean():.1%})")

In [ ]:
# die ersten 100 Intervalle zeichnen, eines pro Zeile; die Fehlgriffe in Rot
fig, ax = plt.subplots(figsize=(8, 7))
for row in range(100):
    colour = "steelblue" if caught[row] else "crimson"
    ax.plot([lowers[row], uppers[row]], [row, row], color=colour, linewidth=2)

ax.axvline(mu, color="black", linestyle="--", label=f"wahrer Mittelwert, {mu} g")
ax.set_title("100 Stichproben, 100 Konfidenzintervalle")
ax.set_xlabel("Körpergewicht (g)")
ax.set_ylabel("Stichprobe")
ax.legend(loc="upper right")
plt.show()

Die meisten Intervalle kreuzen die gestrichelte Linie; eine Handvoll, in Rot, verfehlt sie
komplett. Jedes einzelne wurde auf genau dieselbe Weise gebildet, und von innen sieht man einem
roten Intervall nichts an. **Das sind die 5 Prozent**, und sie verschwinden nie: Sie sind der
Preis dafür, mit einem Bereich zu antworten, statt auf die gesamte Grundgesamtheit zu warten.

Das Niveau ist eine Entscheidung, und zwar ein Kompromiss. Verlangst du 90 Prozent, werden die
Intervalle schmaler, aber etwa eines von zehn liegt daneben; verlangst du 99 Prozent, liegen sie
nur einmal in hundert daneben, sind dafür aber breiter. 95 Prozent ist der übliche Kompromiss.

## 6. Vom Intervall zur Entscheidung

Ein Bestimmungsbuch gibt an, dass Zügelpinguine im Durchschnitt **3.700 g** wiegen. Unsere
Stichprobe hat einen Durchschnitt von 3.733 g. Liegt das Bestimmungsbuch falsch?

Das Intervall beantwortet die Frage. Jeder Wert innerhalb des 95-%-Intervalls ist ein plausibler
wahrer Durchschnitt, einer, den unsere Daten nicht ausschließen können. Jeder Wert außerhalb ist
einer, gegen den die Daten sprechen.

In [ ]:
for claim in [3700, 4000]:
    inside = ci_lower <= claim <= ci_upper
    verdict = "plausibel" if inside else "nicht plausibel"
    print(f"behaupteter Durchschnitt {claim} g: innerhalb von {ci_lower:.0f} bis {ci_upper:.0f}? "
          f"{inside} -> {verdict}")

3.700 g liegt bequem innerhalb des Intervalls, also liegt die Lücke von 33 g zwischen
Bestimmungsbuch und unserer Stichprobe gut im Wackelbereich eines Durchschnitts von 68. Eine
Behauptung von 4.000 g wäre eine ganz andere Geschichte.

Du hast gerade anhand von Daten eine Ja-Nein-Entscheidung über eine Behauptung getroffen, mit
einer bekannten Wahrscheinlichkeit von 5 Prozent, dich täuschen zu lassen. **Das ist ein
Hypothesentest in allem außer dem Namen.** Notebook 03 macht das formal und zeigt, dass es genau
dieselbe Antwort liefert.

> **Übung:**
>
> Kein Code nötig. Ein Gesundheitsamt hat die tägliche Salzaufnahme von 200 Erwachsenen in einer
> Stadt gemessen und gibt ein 95-%-Konfidenzintervall für den Durchschnitt von **2.100 mg bis
> 2.500 mg** an. Der empfohlene Grenzwert liegt bei **2.300 mg**.
>
> 1. Kann das Amt sagen, dass die Erwachsenen der Stadt im Durchschnitt unter dem Grenzwert
>    liegen?
> 2. Kann es sagen, dass der Durchschnitt über dem Grenzwert liegt?
> 3. Schreib einen Satz für den Stadtrat, der sowohl ehrlich als auch nützlich ist.

Deine Antworten hier:

<details><summary>
Hier klicken, um deine Antworten zu vergleichen ...
</summary>

1. **Nein.** Plausible Durchschnitte reichen bis 2.500 mg, also über den Grenzwert.
2. **Nein, das auch nicht.** Plausible Durchschnitte reichen bis hinunter zu 2.100 mg, also unter
   den Grenzwert. Der Grenzwert von 2.300 mg liegt innerhalb des Intervalls, daher können die
   Daten diese beiden Fälle nicht unterscheiden.
3. Zum Beispiel: *„Nach unserer besten Schätzung nehmen Erwachsene hier im Durchschnitt 2.100 bis
   2.500 mg Salz pro Tag auf. Der Durchschnitt kann also auf beiden Seiten des Grenzwerts von
   2.300 mg liegen; um das zu klären, wäre eine größere Studie nötig.“*
</details>

## Zusammenfassung

In diesem Notebook hast du:

- getrennt, was du weißt, die Stichprobenstatistiken, von dem, was du wissen willst, den
  Parametern der Grundgesamtheit.
- beobachtet, wie Stichprobendurchschnitte um den wahren Mittelwert wackeln, und dieses Wackeln
  gemessen: den Standardfehler, $s / \sqrt{n}$.
- gesehen, dass Durchschnitte selbst bei schiefen Daten glockenförmig ausfallen: der zentrale
  Grenzwertsatz.
- ein 95-%-Konfidenzintervall mit der t-Verteilung gebildet und per Simulation überprüft, was die
  95 % wirklich versprechen.
- mit dem Intervall eine Behauptung beurteilt: 3.700 g plausibel, 4.000 g nicht.

Abschnitt 6 hat über die Behauptung des Bestimmungsbuchs anhand eines Intervalls entschieden. In
`03_how_a_test_works.ipynb` stellst du dieselbe Frage andersherum: *Wenn das Bestimmungsbuch recht
hätte, wie überraschend wären dann unsere 68 Pinguine?* Die Antwort ist ein p-Wert.

## Quellen & weiterführende Literatur

- [**scipy.stats.t**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html):
  Die t-Verteilung, einschließlich der in Abschnitt 4 verwendeten Methode `interval`.
- [**Seeing Theory: Frequentist Inference**](https://seeing-theory.brown.edu/frequentist-inference/index.html):
  Eine animierte Version der Simulation aus Abschnitt 5, mit dem Konfidenzniveau als
  Schieberegler.
- [**ModernDive: Confidence Intervals**](https://moderndive.com/v2/confidence-intervals.html):
  Dieselben Ideen, aufgebaut über Resampling – eine gute zweite Erklärung.
- [**StatQuest: Confidence Intervals**](https://www.youtube.com/watch?v=TqOeMYtOc1w):
  Die Idee in unter zehn Minuten.